# CSA0812 – Assignment 3 | Q6
## Nested Data: A Railway Timetable

**Student:** Darshan A  
**Register Number:** 1973260011  
**Marks:** 7 | **Level:** K4 – Analyse

### Aim
Build a nested railway timetable using dictionaries, lists, tuples and sets; query trains by station and running day; sort by departure time; and compare indexed and unindexed queries.

### Prediction
A scan without an index must inspect trains and their stops. A station index narrows the search to trains recorded for that station, so queries should be faster when a station is served by only a small fraction of all trains. The index requires extra memory and update work.

## 1. Main timetable: five trains
Each train number maps to a dictionary containing its name, a list of station tuples `(station, arrival, departure)`, and a set of running days.

In [1]:
timetable = {
    "12601": {
        "name": "Chennai Express",
        "stops": [
            ("Chennai", "06:00", "06:15"),
            ("Vijayawada", "12:00", "12:05"),
            ("Visakhapatnam", "17:30", "17:35"),
            ("Howrah", "05:30", "05:45")
        ],
        "days": {"Mon", "Wed", "Fri"}
    },
    "12602": {
        "name": "Coastal Mail",
        "stops": [
            ("Chennai", "07:00", "07:10"),
            ("Nellore", "09:00", "09:05"),
            ("Vijayawada", "12:30", "12:40"),
            ("Hyderabad", "18:00", "18:10")
        ],
        "days": {"Tue", "Thu", "Sat"}
    },
    "12603": {
        "name": "Southern Link",
        "stops": [
            ("Bengaluru", "05:30", "05:40"),
            ("Katpadi", "08:00", "08:05"),
            ("Chennai", "10:00", "10:15"),
            ("Nellore", "12:30", "12:35")
        ],
        "days": {"Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"}
    },
    "12604": {
        "name": "Deccan Runner",
        "stops": [
            ("Hyderabad", "06:15", "06:25"),
            ("Guntur", "09:00", "09:05"),
            ("Vijayawada", "10:00", "10:10"),
            ("Chennai", "16:00", "16:15")
        ],
        "days": {"Sun"}
    },
    "12605": {
        "name": "Eastern Connector",
        "stops": [
            ("Chennai", "08:30", "08:40"),
            ("Vijayawada", "14:00", "14:05"),
            ("Rajahmundry", "17:00", "17:05"),
            ("Visakhapatnam", "20:00", "20:10")
        ],
        "days": {"Mon", "Thu", "Sat"}
    }
}

def find_trains(data, station, day):
    matches = []
    for train_no, details in data.items():
        if day in details["days"]:
            for stop, arrival, departure in details["stops"]:
                if stop.casefold() == station.casefold():
                    matches.append((train_no, details["name"], departure))
                    break
    return matches

def sort_by_departure(matches):
    return sorted(matches, key=lambda item: item[2])

print("Complete timetable:")
for train_no, details in timetable.items():
    print(f"\\n{train_no} - {details['name']} | Days: {', '.join(sorted(details['days']))}")
    for station, arrival, departure in details["stops"]:
        print(f"  {station:15} Arrival {arrival}  Departure {departure}")

station = "Vijayawada"
day = "Mon"
results = sort_by_departure(find_trains(timetable, station, day))
print(f"\\nTrains stopping at {station} on {day}, sorted by departure:")
for item in results:
    print(item)

# Departure time at the third station (index 2) of a train:
third_station_departure = timetable["12601"]["stops"][2][2]
print("\\nThird station departure for 12601:", third_station_departure)

Complete timetable:
\n12601 - Chennai Express | Days: Fri, Mon, Wed
  Chennai         Arrival 06:00  Departure 06:15
  Vijayawada      Arrival 12:00  Departure 12:05
  Visakhapatnam   Arrival 17:30  Departure 17:35
  Howrah          Arrival 05:30  Departure 05:45
\n12602 - Coastal Mail | Days: Sat, Thu, Tue
  Chennai         Arrival 07:00  Departure 07:10
  Nellore         Arrival 09:00  Departure 09:05
  Vijayawada      Arrival 12:30  Departure 12:40
  Hyderabad       Arrival 18:00  Departure 18:10
\n12603 - Southern Link | Days: Fri, Mon, Sat, Sun, Thu, Tue, Wed
  Bengaluru       Arrival 05:30  Departure 05:40
  Katpadi         Arrival 08:00  Departure 08:05
  Chennai         Arrival 10:00  Departure 10:15
  Nellore         Arrival 12:30  Departure 12:35
\n12604 - Deccan Runner | Days: Sun
  Hyderabad       Arrival 06:15  Departure 06:25
  Guntur          Arrival 09:00  Departure 09:05
  Vijayawada      Arrival 10:00  Departure 10:10
  Chennai         Arrival 16:00  Departure 16:15
\

## 2. Nested structure and data-structure choices

```text
timetable (dictionary)
 ├── "12601" (train number key)
 │    └── train details (dictionary)
 │         ├── name: string
 │         ├── stops: list
 │         │    ├── (station, arrival, departure) tuple
 │         │    └── ...
 │         └── days: set
 └── ...
```

**Expression:** `timetable["12601"]["stops"][2][2]` reaches the departure time at the third station.

- **Outer dictionary:** fast average lookup by train number.
- **Inner dictionary:** named fields keep each train's details organized.
- **List:** preserves the route order of a train's stops.
- **Tuple:** stores the fixed three-part record for one stop.
- **Set:** stores running days without duplicates and supports fast average membership checks.

### Work for one station query
Without an index, the program may inspect every train and every stop. If there are `T` trains and at most `S` stops per train, the worst-case work is **O(T × S)**, plus checking the running-day set. It stops scanning a train after finding the station.

## 3. Build a station index
The index maps each station to train numbers that stop there. The query then checks only candidate trains, verifies the running day, finds the station's departure time and sorts the matches.

In [2]:
def build_station_index(data):
    index = {}
    for train_no, details in data.items():
        for station, arrival, departure in details["stops"]:
            index.setdefault(station.casefold(), []).append(train_no)
    return index

station_index = build_station_index(timetable)

def find_with_index(data, index, station, day):
    matches = []
    for train_no in index.get(station.casefold(), []):
        details = data[train_no]
        if day not in details["days"]:
            continue
        for stop, arrival, departure in details["stops"]:
            if stop.casefold() == station.casefold():
                matches.append((train_no, details["name"], departure))
                break
    return matches

indexed_results = sort_by_departure(find_with_index(timetable, station_index, station, day))
print("Station index:", station_index)
print("Indexed query:", indexed_results)
assert indexed_results == results
print("Indexed and unindexed results match:", True)

Station index: {'chennai': ['12601', '12602', '12603', '12604', '12605'], 'vijayawada': ['12601', '12602', '12604', '12605'], 'visakhapatnam': ['12601', '12605'], 'howrah': ['12601'], 'nellore': ['12602', '12603'], 'hyderabad': ['12602', '12604'], 'bengaluru': ['12603'], 'katpadi': ['12603'], 'guntur': ['12604'], 'rajahmundry': ['12605']}
Indexed query: [('12601', 'Chennai Express', '12:05'), ('12605', 'Eastern Connector', '14:05')]
Indexed and unindexed results match: True


## 4. Experimental investigation: 100, 1,000 and 10,000 trains
Synthetic data is generated with a fixed seed. Each train has a small route selected from a station list and a set of operating days. The benchmark measures repeated queries using the same station and day. The index-building time is separate from query time.

In [3]:
import random
import time
import statistics
import sys

random.seed(24)
STATIONS = ["Chennai", "Vijayawada", "Bengaluru", "Hyderabad",
            "Nellore", "Guntur", "Salem", "Coimbatore",
            "Visakhapatnam", "Rajahmundry", "Katpadi", "Howrah"]
DAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

def make_timetable(train_count):
    data = {}
    for i in range(train_count):
        train_no = f"T{i:05d}"
        route = random.sample(STATIONS, 6)
        stops = []
        for j, place in enumerate(route):
            hour = 5 + j * 2
            stops.append((place, f"{hour:02d}:00", f"{hour:02d}:05"))
        data[train_no] = {
            "name": f"Train {i}",
            "stops": stops,
            "days": set(random.sample(DAYS, 5))
        }
    return data

def timed_query(function, repeats=30):
    samples = []
    for _ in range(repeats):
        start = time.perf_counter()
        answer = function()
        samples.append(time.perf_counter() - start)
    return statistics.median(samples), answer

benchmark = []
for count in (100, 1_000, 10_000):
    data = make_timetable(count)
    build_start = time.perf_counter()
    index = build_station_index(data)
    index_build_time = time.perf_counter() - build_start

    query_station, query_day = "Chennai", "Mon"
    plain_time, plain_answer = timed_query(
        lambda: sort_by_departure(find_trains(data, query_station, query_day))
    )
    index_time, index_answer = timed_query(
        lambda: sort_by_departure(find_with_index(data, index, query_station, query_day))
    )
    assert plain_answer == index_answer

    # Approximate additional memory of the index container and its keys/lists.
    index_memory = sys.getsizeof(index)
    for key, value in index.items():
        index_memory += sys.getsizeof(key) + sys.getsizeof(value)
        index_memory += sum(sys.getsizeof(train_no) for train_no in value)

    benchmark.append({
        "Trains": count,
        "Matching trains": len(plain_answer),
        "Scan query median (s)": plain_time,
        "Indexed query median (s)": index_time,
        "Index build (s)": index_build_time,
        "Index memory estimate (bytes)": index_memory,
        "Results match": True
    })

print("Benchmark (median of 30 queries):")
for row in benchmark:
    print(row)

Benchmark (median of 30 queries):
{'Trains': 100, 'Matching trains': 26, 'Scan query median (s)': 5.6420000007051385e-05, 'Indexed query median (s)': 2.540199999145898e-05, 'Index build (s)': 0.00013431500002525354, 'Index memory estimate (bytes)': 35208, 'Results match': True}
{'Trains': 1000, 'Matching trains': 372, 'Scan query median (s)': 0.0006823245000049383, 'Indexed query median (s)': 0.0003536149999945337, 'Index build (s)': 0.001230096000000458, 'Index memory estimate (bytes)': 333648, 'Results match': True}
{'Trains': 10000, 'Matching trains': 3598, 'Scan query median (s)': 0.00955866249999815, 'Indexed query median (s)': 0.007486821000000532, 'Index build (s)': 0.014271391999983507, 'Index memory estimate (bytes)': 3323616, 'Results match': True}


## 5. Edge cases
An unknown station should return an empty list. A known station queried on a day when a candidate train does not run must exclude that train.

In [4]:
print("Unknown station:", find_with_index(timetable, station_index, "Atlantis", "Mon"))
print("Known station on non-running day:", find_with_index(timetable, station_index, "Vijayawada", "Sun"))

assert find_with_index(timetable, station_index, "Atlantis", "Mon") == []
# Train 12601 stops at Vijayawada but does not run on Sunday.
assert "12601" not in [row[0] for row in find_with_index(timetable, station_index, "Vijayawada", "Sun")]
print("Unknown-station and non-running-day tests passed.")

Unknown station: []
Known station on non-running day: [('12604', 'Deccan Runner', '10:10')]
Unknown-station and non-running-day tests passed.


## 6. Alternative concept: one flat list

A flat representation could store each stop as `(train, station, arrival, departure, days)`:

```python
flat_stops = [
    (train_no, details["name"], station, arrival, departure, details["days"])
    for train_no, details in timetable.items()
    for station, arrival, departure in details["stops"]
]

def flat_query(rows, station, day):
    matches = [
        (train_no, train_name, departure)
        for train_no, train_name, place, arrival, departure, days in rows
        if place.casefold() == station.casefold() and day in days
    ]
    return sorted(matches, key=lambda item: item[2])
```

A flat scan checks every stop for each query: **O(total stops)**, with O(1) extra working space before the result (the returned matches require output space). A station index reduces candidate scanning to **O(stops belonging to trains at that station)**, commonly described as O(trains at the station) when each train has at most one relevant stop, plus sorting the matches. The index adds O(total stops) storage in the worst case because each station-stop association is recorded. The original timetable itself uses O(total stops) storage too.

## 7. Design decision and conclusion

The station index is suitable for frequent station-based enquiry queries because it avoids scanning every train's complete route. The query checks only trains listed for the requested station, then filters by running day and sorts by departure time.

**Trade-off:** the index consumes additional memory and must be rebuilt or updated whenever routes change. For infrequent queries or a very small timetable, a direct scan may be simpler. The benchmark above records the actual execution times and approximate index memory in the running environment; results may vary between machines.

### Complexity summary

| Operation | Time |
|---|---:|
| Build timetable index | O(total stops) |
| Scan without index | O(total stops) |
| Indexed candidate lookup | O(trains serving station × stops per candidate) |
| Sort matching trains | O(k log k), where k is the number of matches |
| Index space | O(total stops) |

### Final verification
The notebook asserts that indexed and unindexed query results match and checks both unknown-station and non-running-day cases.